In [2]:
import joblib

X_train = joblib.load("../data/X_train_raw.joblib")
X_test = joblib.load("../data/X_test_raw.joblib")

y_train = joblib.load("../data/y_train.joblib")
y_test = joblib.load("../data/y_test.joblib")

In [3]:
y_train = y_train.map({
    "Rejected": 0,
    "Approved": 1
})

y_test = y_test.map({
    "Rejected": 0,
    "Approved": 1
})

In [4]:
print(y_train.isnull().sum())
print(y_test.isnull().sum())

0
0


In [5]:
numerical_features = [
    "no_of_dependents",
    " income_annum",
    " loan_amount",
    " loan_term",
    " cibil_score",
    " residential_assets_value",
    " commercial_assets_value",
    " luxury_assets_value",
    " bank_asset_value",
    " total_assets",
    " loan_to_income_ratio",
    " asset_to_loan_ratio",
    " asset_to_income_ratio"
]

categorical_features = [
    " education",
    " self_employed"
]

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [7]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [8]:
from xgboost import XGBClassifier

In [9]:
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

In [10]:
from sklearn.pipeline import Pipeline

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

In [11]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [12]:
from sklearn.model_selection import cross_validate

xgb_cv_results = cross_validate(
    xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    return_train_score=True,
    n_jobs=-1
)

In [13]:
print("Mean CV Accuracy :",
      xgb_cv_results["test_accuracy"].mean())

print("Mean CV Precision:",
      xgb_cv_results["test_precision"].mean())

print("Mean CV Recall   :",
      xgb_cv_results["test_recall"].mean())

print("Mean CV F1       :",
      xgb_cv_results["test_f1"].mean())

print("Mean CV ROC-AUC  :",
      xgb_cv_results["test_roc_auc"].mean())

Mean CV Accuracy : 0.9970717423133235
Mean CV Precision: 0.9971863970709502
Mean CV Recall   : 0.9981176470588234
Mean CV F1       : 0.9976492642453543
Mean CV ROC-AUC  : 0.9999416324669402


In [14]:
print("Mean Train Accuracy :",
      xgb_cv_results["train_accuracy"].mean())

print("Mean Train Precision:",
      xgb_cv_results["train_precision"].mean())

print("Mean Train Recall   :",
      xgb_cv_results["train_recall"].mean())

print("Mean Train F1       :",
      xgb_cv_results["train_f1"].mean())

print("Mean Train ROC-AUC  :",
      xgb_cv_results["train_roc_auc"].mean())

Mean Train Accuracy : 1.0
Mean Train Precision: 1.0
Mean Train Recall   : 1.0
Mean Train F1       : 1.0
Mean Train ROC-AUC  : 1.0


In [15]:
# Tune n_estimators

In [16]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [50, 100, 150, 200, 300]
}

xgb_grid_estimators = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

xgb_grid_estimators.fit(X_train, y_train)

print("Best n_estimators:", xgb_grid_estimators.best_params_)
print("Best CV ROC-AUC:", xgb_grid_estimators.best_score_)

Best n_estimators: {'model__n_estimators': 50}
Best CV ROC-AUC: 0.9999434564523485


In [17]:
# Tune max_depth

In [18]:
param_grid = {
    "model__n_estimators": [50],
    "model__max_depth": [2, 3, 4, 5, 6, 8, 10]
}

xgb_grid_depth = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

xgb_grid_depth.fit(X_train, y_train)

print("Best parameters:", xgb_grid_depth.best_params_)
print("Best CV ROC-AUC:", xgb_grid_depth.best_score_)

Best parameters: {'model__max_depth': 4, 'model__n_estimators': 50}
Best CV ROC-AUC: 0.9999471044231647


In [19]:
# Tune learning_rate

In [20]:
param_grid = {
    "model__n_estimators": [50],
    "model__max_depth": [4],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2]
}

xgb_grid_lr = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

xgb_grid_lr.fit(X_train, y_train)

print("Best parameters:", xgb_grid_lr.best_params_)
print("Best CV ROC-AUC:", xgb_grid_lr.best_score_)

Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__n_estimators': 50}
Best CV ROC-AUC: 0.999938896488828


In [21]:
param_grid = {
    "model__n_estimators": [50],
    "model__max_depth": [4],
    "model__learning_rate": [0.05, 0.1, 0.2, 0.3]
}

xgb_grid_lr = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

xgb_grid_lr.fit(X_train, y_train)

print("Best parameters:", xgb_grid_lr.best_params_)
print("Best CV ROC-AUC:", xgb_grid_lr.best_score_)

Best parameters: {'model__learning_rate': 0.3, 'model__max_depth': 4, 'model__n_estimators': 50}
Best CV ROC-AUC: 0.9999471044231647


In [22]:
# tune min_child_weight

In [23]:
param_grid = {
    "model__min_child_weight": [1, 3, 5, 7, 10]
}

grid_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

Best parameters: {'model__min_child_weight': 3}
Best CV ROC-AUC: 0.9999471044231647


In [24]:
# Tune subsample

In [25]:
param_grid = {
    "model__n_estimators": [50],
    "model__max_depth": [4],
    "model__learning_rate": [0.3],
    "model__min_child_weight": [3],
    "model__subsample": [0.6, 0.8, 1.0]
}

xgb_grid_subsample = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

xgb_grid_subsample.fit(X_train, y_train)

print("Best parameters:", xgb_grid_subsample.best_params_)
print("Best CV ROC-AUC:", xgb_grid_subsample.best_score_)

Best parameters: {'model__learning_rate': 0.3, 'model__max_depth': 4, 'model__min_child_weight': 3, 'model__n_estimators': 50, 'model__subsample': 1.0}
Best CV ROC-AUC: 0.99993707250342


In [26]:
# Tune colsample_bytree

In [27]:
param_grid = {
    "model__n_estimators": [50],
    "model__max_depth": [4],
    "model__learning_rate": [0.3],
    "model__min_child_weight": [3],
    "model__subsample": [1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0]
}

xgb_grid_colsample = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

xgb_grid_colsample.fit(X_train, y_train)

print("Best parameters:", xgb_grid_colsample.best_params_)
print("Best CV ROC-AUC:", xgb_grid_colsample.best_score_)

Best parameters: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.3, 'model__max_depth': 4, 'model__min_child_weight': 3, 'model__n_estimators': 50, 'model__subsample': 1.0}
Best CV ROC-AUC: 0.9999516643866849


# final pipeline

In [28]:
xgb_final = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=50,
        max_depth=4,
        learning_rate=0.3,
        min_child_weight=3,
        subsample=1.0,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    ))
])

xgb_final.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [29]:
y_pred_xgb = xgb_final.predict(X_test)

y_prob_xgb = xgb_final.predict_proba(X_test)[:, 1]

In [30]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("XGBoost Test Accuracy :",
      accuracy_score(y_test, y_pred_xgb))

print("XGBoost Test Precision:",
      precision_score(y_test, y_pred_xgb))

print("XGBoost Test Recall   :",
      recall_score(y_test, y_pred_xgb))

print("XGBoost Test F1       :",
      f1_score(y_test, y_pred_xgb))

print("XGBoost Test ROC-AUC  :",
      roc_auc_score(y_test, y_prob_xgb))

XGBoost Test Accuracy : 0.9976580796252927
XGBoost Test Precision: 0.9981167608286252
XGBoost Test Recall   : 0.9981167608286252
XGBoost Test F1       : 0.9981167608286252
XGBoost Test ROC-AUC  : 0.9999825086145074


In [31]:
from sklearn.metrics import confusion_matrix

cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print(cm_xgb)

[[322   1]
 [  1 530]]


In [32]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       323
           1       1.00      1.00      1.00       531

    accuracy                           1.00       854
   macro avg       1.00      1.00      1.00       854
weighted avg       1.00      1.00      1.00       854



In [33]:
print(xgb_grid_colsample.best_estimator_)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['no_of_dependents',
                                                   ' income_annum',
                                                   ' loan_amount', ' loan_term',
                                                   ' cibil_score',
                                                   ' residential_assets_value',
                                                   ' commercial_assets_value',
                                                   ' luxury_assets_value',
                                                   ' bank_asset_value',
                                                   ' total_assets',
                                                   ' loan_to_income_ratio',
                                                   ' asset_to_loan_ratio',
                                                   ' asset_to_income_ratio']),
 

In [35]:
import joblib

In [36]:
joblib.dump(
    xgb_grid_colsample.best_estimator_,
    "../models/xgboost_final_pipeline.pkl"
)

['../models/xgboost_final_pipeline.pkl']

In [37]:
import os

print(
    os.path.exists(
        "../models/xgboost_final_pipeline.pkl"
    )
)

True


In [38]:
file_path = "../models/xgboost_final_pipeline.pkl"

print(
    os.path.getsize(file_path),
    "bytes"
)

65671 bytes
